In [32]:
# Importing the Dependencies

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import re
import nltk
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords  # corpus means some text content and nltk means natural language tool kit
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abbas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [34]:
# Printing the stopwords
print(stopwords.words('english'))  # These words are not usefull for us or meaning less for our use case

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# Data Preprocessing


In [35]:
# Load the data to a pandas DataFrame

news_data = pd.read_csv('fake_news_dataset.csv')

In [36]:
# First Five rows

news_data.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


- 0 --> Fake News
- 1 --> Real News

In [37]:
news_data.shape

(20800, 5)

In [38]:
# Checking Missing Values

news_data.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [39]:
# Replace Missing Values with Empty String

news_data = news_data.fillna('')

In [40]:
# Rechecking Missing values

news_data.isnull().sum()

id        0
title     0
author    0
text      0
label     0
dtype: int64

In [41]:
# Lets Merge Author name and News Title 

news_data['content'] = news_data['author']+' '+news_data['title']

In [42]:
# Printing fist five rows

news_data.head()

,id,title,author,text,label,content
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1,Consortiumnews.com Why the Truth Might Get You...
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1,Jessica Purkiss 15 Civilians Killed In Single ...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1,Howard Portnoy Iranian woman jailed for fictio...


# Seperating Features and Target

In [43]:
X = news_data.drop(columns='label', axis=1)
Y = news_data['label']


In [44]:
print(X)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

In [45]:
print(Y)

0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 20800, dtype: int64


# Stemming

- Stemming is the process of reducing a word to its Root Word

    - Example: The words "running," "runs," and "runner" can all be stemmed to "run". 

In [46]:
port_stem = PorterStemmer()

In [47]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    return stemmed_content


In [48]:
news_data['content'] = news_data['content'].apply(stemming)

In [49]:
print(news_data['content'])

0        [darrel, lucu, hous, dem, aid, even, see, come...
1        [daniel, j, flynn, flynn, hillari, clinton, bi...
2            [consortiumnew, com, truth, might, get, fire]
3        [jessica, purkiss, civilian, kill, singl, us, ...
4        [howard, portnoy, iranian, woman, jail, fictio...
                               ...                        
20795    [jerom, hudson, rapper, trump, poster, child, ...
20796    [benjamin, hoffman, n, f, l, playoff, schedul,...
20797    [michael, j, de, la, merc, rachel, abram, maci...
20798    [alex, ansari, nato, russia, hold, parallel, e...
20799                      [david, swanson, keep, f, aliv]
Name: content, Length: 20800, dtype: object
